# Module 04: Seaborn for Machine Learning
## Notebook 01: Seaborn Architecture and Distribution Visualizations

Seaborn is a high-level statistical data visualization library built on top of Matplotlib and integrated tightly with Pandas DataFrames. It automates complex statistical aggregations, error estimation, and multi-dimensional aesthetic mappings with minimal code.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the architectural difference between **Figure-Level** and **Axes-Level** functions.
2. Apply global statistical themes and palettes using `sns.set_theme()`.
3. Visualize univariate continuous distributions with `sns.histplot()` and Kernel Density Estimation (`kde=True`).
4. Generate bivariate density surfaces using `sns.kdeplot()`.
5. Quantify cumulative probability thresholds using `sns.ecdfplot()`.
6. **Advanced:** Construct multi-faceted conditioning layouts using Figure-Level `sns.displot()` with bivariate contour overlays and empirical quantile estimations.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set consistent, modern styling
sns.set_theme(style="whitegrid", palette="muted")
print(f"Seaborn version: {sns.__version__}")

### 1. Figure-Level vs. Axes-Level Architecture

Understanding this distinction avoids 90% of Seaborn layout confusion:
- **Axes-Level Functions** (e.g., `histplot`, `kdeplot`, `scatterplot`, `boxplot`):
  - Draw directly onto an existing Matplotlib `Axes` object.
  - Accept an `ax=...` argument and integrate seamlessly into Matplotlib multi-subplot grids!
- **Figure-Level Functions** (e.g., `displot`, `relplot`, `catplot`):
  - Control their own whole `Figure` (instantiating a `FacetGrid`).
  - Cannot be passed an existing `ax` argument directly.

In [ ]:
# Generate synthetic classification dataset: Two customer age distributions
rng = np.random.default_rng(42)
ages_churned = rng.normal(loc=32, scale=7, size=300)
ages_retained = rng.normal(loc=46, scale=9, size=700)

df = pd.DataFrame({
    'Age': np.concatenate([ages_churned, ages_retained]),
    'Customer_Status': ['Churned'] * len(ages_churned) + ['Retained'] * len(ages_retained)
})

# Axes-level integration with Matplotlib
fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.histplot(
    data=df, 
    x='Age', 
    hue='Customer_Status', 
    kde=True, 
    bins=30, 
    palette={'Churned': '#E15759', 'Retained': '#4E79A7'},
    alpha=0.6,
    ax=ax
)

ax.set_title("Customer Age Distribution Stratified by Churn Status", fontsize=13, fontweight='bold')
ax.set_xlabel("Customer Age (Years)")
ax.set_ylabel("Frequency Count")

plt.show()

---
### 2. Kernel Density Estimation (KDE) and Bivariate Surfaces

KDE estimates the continuous Probability Density Function (PDF) of a random variable without assuming a parametric form (like Gaussian).
Bivariate KDE reveals multi-modal clusters and non-linear joint interactions.

In [ ]:
# Generate 2 continuous features
x_feat = rng.normal(0, 1, 400)
y_feat = x_feat * 0.7 + rng.normal(0, 0.6, 400)
df_bi = pd.DataFrame({'Feature_1': x_feat, 'Feature_2': y_feat})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# 1D KDE with rug plot
sns.kdeplot(data=df_bi, x='Feature_1', fill=True, color='purple', ax=ax1)
sns.rugplot(data=df_bi, x='Feature_1', color='darkmagenta', height=0.08, ax=ax1)
ax1.set_title("1D KDE with Marginal Rug Plot", fontsize=12, fontweight='bold')

# 2D Bivariate density contour surface
sns.kdeplot(data=df_bi, x='Feature_1', y='Feature_2', fill=True, cmap='mako', levels=8, thresh=0.05, ax=ax2)
ax2.scatter(df_bi['Feature_1'], df_bi['Feature_2'], s=10, alpha=0.3, color='black')
ax2.set_title("2D Bivariate Density Surface", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

---
### 3. Empirical Cumulative Distribution Functions: `sns.ecdfplot()`

While histograms and KDEs depend heavily on arbitrary bin-widths or smoothing bandwidths:
- **ECDF** has **zero tuning parameters**: it displays the exact proportion of observations less than or equal to $x$:
$$F_n(x) = \frac{1}{n} \sum_{i=1}^n \mathbb{I}(x_i \le x)$$
- Allows instant evaluation of percentiles (e.g., median, 90th percentile, 99th percentile).

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.ecdfplot(data=df, x='Age', hue='Customer_Status', palette={'Churned': '#E15759', 'Retained': '#4E79A7'}, linewidth=2, ax=ax)

ax.set_title("Empirical Cumulative Distribution Function (ECDF)", fontsize=13, fontweight='bold')
ax.set_xlabel("Age")
ax.set_ylabel("Cumulative Probability P(Age <= x)")
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7)

plt.show()

---
### 4. Advanced Complex Usage: Multi-Faceted Conditioning with Figure-Level `sns.displot`

In high-dimensional datasets, distributions change conditioned across multiple categorical variables (e.g. Age distribution conditioned on Subscription Tier AND Churn Status).

Using Figure-Level `sns.displot`:
- We map `col='Tier'`, `hue='Customer_Status'`, and `kind='kde'` simultaneously.
- Seaborn automatically handles facet subplot creation, shared axes, and clean unified legends.

In [ ]:
# Generate multi-tiered customer cohort data
n_records = 1200
cohort_df = pd.DataFrame({
    'Tenure_Months': np.concatenate([
        rng.exponential(scale=8, size=400),
        rng.normal(loc=24, scale=6, size=400),
        rng.normal(loc=48, scale=10, size=400)
    ]),
    'Tier': ['Bronze'] * 400 + ['Silver'] * 400 + ['Gold'] * 400,
    'Contract': rng.choice(['Month-to-Month', 'Annual'], size=n_records, p=[0.6, 0.4])
})
# Introduce churn status correlated with tier and tenure
cohort_df['Churn'] = np.where(cohort_df['Tenure_Months'] < 15, 'Yes', 'No')

# Multi-faceted conditional displot across columns and hue
g = sns.displot(
    data=cohort_df,
    x='Tenure_Months',
    hue='Churn',
    col='Tier',
    row='Contract',
    kind='kde',
    fill=True,
    common_norm=False,
    palette={'Yes': '#e74c3c', 'No': '#2ecc71'},
    height=3.0,
    aspect=1.2
)

g.set_axis_labels("Tenure (Months)", "Density")
g.fig.subplots_adjust(top=0.90)
g.fig.suptitle("Faceted Conditioning: Tenure Distribution by Tier, Contract, and Churn", fontsize=13, fontweight='bold')

plt.show()

### Summary & Next Steps
In this notebook, you mastered:
- Figure-Level vs. Axes-Level architectural principles in Seaborn.
- Univariate and bivariate continuous density estimations (`histplot`, `kdeplot`).
- ECDF analysis for parameter-free percentile quantification.
- Multi-faceted conditional distribution grids with Figure-Level `displot()`.

**Next Notebook:** `02_relational_and_categorical_plots.ipynb` — Relational mappings with hue/style/size, bootstrap confidence intervals, and categorical distributions.